# Setup

In [45]:
from statasetup import *

In [46]:
%%stata
use "Datasets\asg1\Stereotype_threat.dta",clear

# Between-Subjects Anova with contrast analysis follow up
Be the contrast analysis the desired post-hoc analysis technique one should always try to match a **1 by n** design, diverting a bit from the initial study design.

This is usually done by using `group`. Doing it so will result in a less comprehensive answer but efortless generation of **SScontrast**

## 1. Conduct ANOVA with grouped effects

In [53]:
%%stata
anova performance race##test_type


                         Number of obs =         20    R-squared     =  0.5160
                         Root MSE      =    2.91548    Adj R-squared =  0.4253

                  Source | Partial SS         df         MS        F    Prob>F
          ---------------+----------------------------------------------------
                   Model |        145          3   48.333333      5.69  0.0076
                         |
                    race |         20          1          20      2.35  0.1446
               test_type |         45          1          45      5.29  0.0352
          race#test_type |         80          1          80      9.41  0.0074
                         |
                Residual |        136         16         8.5  
          ---------------+----------------------------------------------------
                   Total |        281         19   14.789474  


In [39]:
%%stata
drop groups
egen groups = group(race test_type)
anova performance groups

* Ideally, SS_model must be calculated and stored in mata's memory right away to prevent its loss
mata
SSmodel = st_numscalar("e(mss)")
SSmodel
end



. drop groups

. egen groups = group(race test_type)

. anova performance groups

                         Number of obs =         20    R-squared     =  0.5160
                         Root MSE      =    2.91548    Adj R-squared =  0.4253

                  Source | Partial SS         df         MS        F    Prob>F
              -----------+----------------------------------------------------
                   Model |        145          3   48.333333      5.69  0.0076
                         |
                  groups |        145          3   48.333333      5.69  0.0076
                         |
                Residual |        136         16         8.5  
              -----------+----------------------------------------------------
                   Total |        281         19   14.789474  

. 
. * Ideally, SS_model must be calculated and stored in mata's memory right away
>  to prevent its loss
. mata
------------------------------------------------- mata (type end to e

In [40]:
%%stata
matrix L =(3, 1/3, 1/3, 1/3, 0)
manova performance = groups
manovatest, test(L)

* Same as before but for this time the contrast makes the item to be stored
mata
SScontrast = st_matrix("r(H)")
SScontrast
end


. matrix L =(3, 1/3, 1/3, 1/3, 0)

. manova performance = groups

                       Number of obs =         20

                       W = Wilks' lambda      L = Lawley-Hotelling trace
                       P = Pillai's trace     R = Roy's largest root

              Source | Statistic        df    F(df1,     df2) =   F   Prob>F
          -----------+-------------------------------------------------------
              groups |W   0.4840         3      3.0     16.0     5.69 0.0076 e
                     |P   0.5160                3.0     16.0     5.69 0.0076 e
                     |L   1.0662                3.0     16.0     5.69 0.0076 e
                     |R   1.0662                3.0     16.0     5.69 0.0076 e
                     |-------------------------------------------------------
            Residual |                  16
          -----------+-------------------------------------------------------
               Total |                  19
          ----------------

## 2. Calculate eta alerting
**All the aforementioned steps must have been taken in order to have mata script able to calculate eta2alertin correctly**

In [41]:
%%stata

mean performance, over(groups)

// finally use mata to calculate the C-estimate and the eta squared alerting
mata
Means = st_matrix("e(b)")
L = st_matrix("L") 
Cest = rowsum(L[1,1::(cols(L) - 1)] :* Means) 	  
Cest	
Eta2alerting = SScontrast/SSmodel
Eta2alerting				
end


. 
. mean performance, over(groups)

Mean estimation                                     Number of obs = 20

----------------------------------------------------------------------
                     |       Mean   Std. err.     [95% conf. interval]
---------------------+------------------------------------------------
c.performance@groups |
                  1  |          4    1.30384      1.271031    6.728969
                  2  |         11    1.30384      8.271031    13.72897
                  3  |         10    1.30384      7.271031    12.72897
                  4  |          9    1.30384      6.271031    11.72897
----------------------------------------------------------------------

. 
. // finally use mata to calculate the C-estimate and the eta squared alerting
. mata
------------------------------------------------- mata (type end to exit) -----
: Means = st_matrix("e(b)")

: L = st_matrix("L") 

: Cest = rowsum(L[1,1::(cols(L) - 1)] :* Means)     

: Cest    
  22

: Eta2